## Create Table to summarize results

This notebook creates a table with the normalized NLLHs for different models. It expects to find pickled predictions for 10 folds with the following filenames:

 * ```<scenario_name>.floc.<dist_name>.<fold>.100.<model>.pkl```

`eval_model.py` automatically creates such files. 

**NOTE:** The first cell contains some variables that need to be adjusted. Evaluating all scenarios might take a while.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import tabulate

#% matplotlib inline

sys.path.append("../")
from helper_DistNet import load_data_continuous, preprocess

mean_exp = []  #["rf", "nn"]

# 1) Select the distribution you want to fit
floc_exp = ["lognormal_xgb"]
distribution_hndl = scst.distributions.lognorm
# floc_exp = ["expon_xgb"]
# distribution_hndl = scst.distributions.expon
# floc_exp = ["invgauss_xgb"]
# distribution_hndl = scst.distributions.invgauss
# floc_exp = ["weibull_xgb"]
# distribution_hndl = scst.distributions.weibull_min

# 2) Change the following to existing paths
output_dirs = [
    "../scripts_DistNet/TEST_50_SMAC_NAS_500/", # SMAC with 500 trials - NAS configspace
    "../scripts_DistNet/TEST_50_SMAC_NAS_100/", # SMAC with 100 trials - NAS configspace
]
save_dirs = [
    "./results_eval_SMAC_NAS_500/",
    "./results_eval_SMAC_NAS_100/",
]
conditions = [
    "SMAC_NAS_500",
    "SMAC_NAS_100",
]
save_dir_combined = "../results_combined/"
os.makedirs(save_dir_combined, exist_ok=True)

# 3) Select whether to compute results for all scenarios or just a single scenario
SCENARIOS = ["DE", "DiagonalCMA", "modcma", "modde", "NaiveIsoEMNA", "NGOpt14", "NGOpt38", "OnePlusOne", "PSO", "RandomSearch", "RCobyla"]
DIMS =  [5, 20]
BUDGETS = [100, 250, 500, 1000] # , 2500, 5000, 10000]
# BUDGETS = [100, 10000]
RUNS = 50
LOG_TARGET = False

load = True

In [ ]:
dir_res_dict = dict()

for output_dir in output_dirs:
    res_dict = dict()  
    if not os.path.isdir(output_dir):
        print("Directory does not exist: %s" % output_dir)
        continue
    print("Processing directory: %s" % output_dir)
    for fl in os.listdir(output_dir):
        if not (f".50." in fl or f".1." in fl):
            # Select only one seed
            continue
        print(fl, output_dir)

        fl_path = os.path.join(output_dir, fl)
        
        if os.path.isdir(fl_path):
            continue
        
        if not "pkl" in fl_path:
            continue
        
        if "txt" in fl_path or "png" in fl_path:
            continue

        if not os.path.isfile(fl_path):
            fl_path = os.path.join(output_dir)

        with open(fl_path, "rb") as fh:
            pkl = pickle.load(fh)
            train_pred = np.array(pkl[0])
            val_pred = np.array(pkl[1])
            add_info = pkl[2]
            task = add_info["task"]
            model = add_info["model"]
            scenario = add_info["scenario"]
            dim = add_info["dim"]
            budget = add_info["budget"]
            fold = add_info["fold"]

            # Flip the prediction 
            if 'xgb' in model:
                train_pred = train_pred[:, ::-1]
                val_pred = val_pred[:, ::-1]
                # Convert log of location -> location
                if 'lognormal' in model:
                    train_pred[:, 1] = np.exp(train_pred[:, 1])
                    val_pred[:, 1] = np.exp(val_pred[:, 1])

            if "nn" in add_info["model"] and add_info["task"] == "floc":
                print(f"{scenario}_{dim}D_{budget}, {fold}, {add_info['loaded']}")
        
        SC = f"{scenario}_{dim}D_{budget}"
        if SC not in res_dict:
            res_dict[SC] = dict()

        if model not in floc_exp and model not in mean_exp:
            continue
        
        if task not in res_dict[SC]:
            res_dict[SC][task] = dict()
            
        if model not in res_dict[SC][task]:
            res_dict[SC][task][model] = dict()

        res_dict[SC][task][model][fold] = (train_pred, val_pred)

    # Store results for this directory
    dir_res_dict[output_dir] = res_dict

In [ ]:
dir_nllh_dict = dict()
dir_mean_dict = dict()
dir_var_dict = dict()

for save_dir in save_dirs:
    nllh_dict = dict()
    mean_dict = dict()
    var_dict = dict()
    if load:
        with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
            nllh_dict = pickle.load(fh)
            
        with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
            mean_dict = pickle.load(fh)
            
        with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
            var_dict = pickle.load(fh)
    dir_nllh_dict[save_dir] = nllh_dict
    dir_mean_dict[save_dir] = mean_dict
    dir_var_dict[save_dir] = var_dict
    print(nllh_dict.keys())
        
    print(f"LOADED: {save_dir}")


In [ ]:
# Generate NLLH table

# nan means there are some folds missing
# inf means there was no results file found
tab_data = list()
header = ["sc"] + [sd[-3:] for sd in conditions]
for scen in sorted(nllh_dict.keys(), key=lambda x: int(x.split("_")[2])):
    dummy_array = np.ones([2, 2]) * np.inf
    tab_data.append([scen] + [np.mean(nllh_dict[scen].get(floc_exp[0], dummy_array)[:, 0]) for nllh_dict in dir_nllh_dict.values()])
    tab_data.append(["", ] + [np.mean(nllh_dict[scen].get(floc_exp[0], dummy_array)[:, 1]) for nllh_dict in dir_nllh_dict.values()])

a = tabulate.tabulate(tab_data, headers=header, floatfmt="5.3f")
print(a)

with open("%s/nllh_%s.txt" % (save_dir_combined, distribution_hndl.name), "w") as fh:
    fh.write(a)

In [ ]:
model_name = {
    "lognormal_distfit": "fitted",
    "lognormal_distfit.rf2": "iRF",
    "lognormal_distfit.rf": "mRF",
    "lognormal_nn": "DistNet",
    "lognormal_xgb": "XGBoostLSS",

    "invgauss_distfit": "fitted",
    "invgauss_distfit.rf2": "iRF",
    "invgauss_distfit.rf": "mRF",
    "invgauss_nn": "DistNet",
    "invgauss_xgb": "XGBoostLSS",

    "weibull_distfit": "fitted",
    "weibull_distfit.rf2": "iRF",
    "weibull_distfit.rf": "mRF",
    "weibull_nn": "DistNet",
    "weibull_xgb": "XGBoostLSS",
}

algo_name = {
    "DE": "DE",
    "DiagonalCMA": "Diagonal CMA",
    "modcma": "Mod CMA",
    "modde": "L-SHADE",
    "NaiveIsoEMNA": "NaiveIsoEMNA",
    "NGOpt14": "NGOpt14",
    "NGOpt38": "NGOpt38",
    "OnePlusOne": "OnePlusOne",
    "PSO": "PSO",
    "RandomSearch": "Random Search",
    "RCobyla": "RCobyla",
}

for algo in SCENARIOS:
    # algo = algo[0] # Choose one algorithm to create table for
    # algo = "RandomSearch"
    tabular_data = list()
    TAB_SCENARIOS = list()
    CAP_NUM = 1e10

    for dim in DIMS:
        for budget in BUDGETS:
            TAB_SCENARIOS.append(f"{algo}_{dim}D_{budget}")

    header = ["Dimension"] + ["Budget"] + [""] + floc_exp
    # header = [""] + floc_exp
    header_start_index = len(header) - len(floc_exp)
    for scen in TAB_SCENARIOS:
        dummy_array = np.ones([2, 2]) * np.inf
        tabular_data.append(
            [f"{scen.split('_')[1]}", f"{scen.split('_')[2]}"] + 
            ["train"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) 
            if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
            else np.inf for m in header[header_start_index:] for nllh_dict in dir_nllh_dict.values()]
        )
        tabular_data.append(
            ["", "" ] + 
            ["test"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) 
            if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
            else np.inf for m in header[header_start_index:] for nllh_dict in dir_nllh_dict.values()]
        )
    latex_header = header[:header_start_index] + [model_name[m] for m in floc_exp]

    latex_tabular = tabulate.tabulate(tabular_data, tablefmt="latex", floatfmt="5.3f", headers=latex_header)
    # print(latex_tabular)

    os.makedirs(f"{save_dir_combined}/{floc_exp[0].split('_')[0]}/", exist_ok=True)
    with open(f"{save_dir_combined}/{floc_exp[0].split('_')[0]}/{algo}.tex", "w") as fh:
        fh.write(latex_tabular)